# Streaming Content Intelligence

This notebook analyzes the Netflix catalog as a content intelligence problem. The analysis focuses on catalog composition, country and genre concentration, platform timing, description-driven clustering, and title-type prediction.

## Questions

- How is the catalog split between movies and TV shows?
- Which countries and genres dominate the catalog?
- How long after release are titles typically added?
- Are rating distributions meaningfully different between movies and TV shows?
- Can description text reveal natural content clusters?
- Can simple metadata predict whether a title is a movie or TV show?

In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = ROOT / 'data' / 'raw' / 'netflix_titles.csv'
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.mplconfig'))
(ROOT / '.mplconfig').mkdir(exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 140
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.titleweight'] = 'bold'

In [ ]:
df = pd.read_csv(DATA_PATH)
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['rating'] = df['rating'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['duration'] = df['duration'].fillna('Unknown')
df['date_added_year'] = df['date_added'].dt.year
df['date_added_month'] = df['date_added'].dt.to_period('M').dt.to_timestamp()
df['primary_country'] = df['country'].str.split(',').str[0].str.strip()
df['primary_genre'] = df['listed_in'].str.split(',').str[0].str.strip()
df['cast_size'] = df['cast'].apply(lambda x: 0 if x == 'Unknown' else len([c for c in x.split(',') if c.strip()]))
df['description_length'] = df['description'].fillna('').str.split().str.len()
df['content_age_at_add'] = np.where(df['date_added_year'].notna(), df['date_added_year'] - df['release_year'], np.nan)
df.shape

In [ ]:
summary = {
    'catalog_titles': int(len(df)),
    'movies': int((df['type'] == 'Movie').sum()),
    'tv_shows': int((df['type'] == 'TV Show').sum()),
    'countries_covered': int(df['primary_country'].nunique()),
    'genres_covered': int(df['primary_genre'].nunique()),
    'catalog_start_year': int(df['release_year'].min()),
    'catalog_end_year': int(df['release_year'].max()),
    'average_description_length': round(float(df['description_length'].mean()), 2),
    'median_content_age_at_add': round(float(df['content_age_at_add'].dropna().median()), 2),
}
summary

In [ ]:
timeline = df.dropna(subset=['date_added_month']).groupby(['date_added_month', 'type']).size().reset_index(name='titles')
plt.figure(figsize=(12, 6))
sns.lineplot(data=timeline, x='date_added_month', y='titles', hue='type', linewidth=3)
plt.title('Catalog Additions Over Time')
plt.xlabel('Month Added')
plt.ylabel('Titles Added')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
country_df = (
    df.groupby('primary_country')
    .agg(Titles=('show_id', 'count'))
    .sort_values('Titles', ascending=False)
    .reset_index()
)
top_country = country_df.head(12).sort_values('Titles')
plt.figure(figsize=(12, 7))
plt.barh(top_country['primary_country'], top_country['Titles'], color=sns.color_palette('viridis', len(top_country)))
plt.title('Top Countries by Catalog Size')
plt.xlabel('Titles')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

In [ ]:
genre_df = (
    df.groupby('primary_genre')
    .agg(Titles=('show_id', 'count'))
    .sort_values('Titles', ascending=False)
    .reset_index()
)
top_genres = genre_df.head(12).sort_values('Titles')
plt.figure(figsize=(12, 7))
plt.barh(top_genres['primary_genre'], top_genres['Titles'], color=sns.color_palette('rocket', len(top_genres)))
plt.title('Top Primary Genres in the Catalog')
plt.xlabel('Titles')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

In [ ]:
rating_mix = df[df['rating'] != 'Unknown'].groupby(['rating', 'type']).size().reset_index(name='titles')
top_ratings = rating_mix.groupby('rating')['titles'].sum().nlargest(10).index
subset = rating_mix[rating_mix['rating'].isin(top_ratings)]
pivot = subset.pivot(index='rating', columns='type', values='titles').fillna(0).sort_values('Movie')
pivot.plot(kind='barh', stacked=True, figsize=(12, 6), color=['#4c78a8', '#f58518'])
plt.title('Content Rating Mix by Title Type')
plt.xlabel('Titles')
plt.ylabel('Rating')
plt.tight_layout()
plt.show()

In [ ]:
chi2, p_value, _, _ = stats.chi2_contingency(pd.crosstab(df['type'], df['rating']))
{'rating_type_p_value': p_value}

In [ ]:
text = df['title'].fillna('') + ' ' + df['listed_in'].fillna('') + ' ' + df['description'].fillna('')
vectorizer = TfidfVectorizer(stop_words='english', max_features=1200, min_df=5)
matrix = vectorizer.fit_transform(text)
kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
df['cluster_id'] = kmeans.fit_predict(matrix)
terms = np.array(vectorizer.get_feature_names_out())

cluster_rows = []
for idx in range(6):
    center = kmeans.cluster_centers_[idx]
    top_terms = ', '.join(terms[center.argsort()[-8:]][::-1])
    subset = df[df['cluster_id'] == idx]
    cluster_rows.append({
        'cluster_id': idx,
        'titles': len(subset),
        'dominant_type': subset['type'].mode().iloc[0],
        'top_terms': top_terms,
    })

cluster_df = pd.DataFrame(cluster_rows).sort_values('titles', ascending=False)
cluster_df

In [ ]:
feature_df = df[['primary_country', 'rating', 'description_length', 'cast_size', 'release_year']].copy()
target = df['type']
X_train, X_test, y_train, y_test = train_test_split(feature_df, target, test_size=0.2, random_state=42, stratify=target)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), ['description_length', 'cast_size', 'release_year']),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), ['primary_country', 'rating']),
    ]
)

model = Pipeline([
    ('preprocess', preprocessor),
    ('clf', LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
{'accuracy': accuracy_score(y_test, predictions)}

## Conclusion

This project treats catalog analysis as an intelligence problem rather than a simple dashboard exercise. The combination of statistical testing, text clustering, and metadata-based modeling makes the repository more aligned with modern analytical and modeling workflows.